# 08 · Summary statistics

The summary statistics of the flux, built bottom-up on toy boxes whose statistics are known: the 3D fields and their
statistics (Part A), skewers through them, from the gas to the flux (Part B), the 3D statistics of the overfluxes on the
lattice of skewers (Part C), and the 1D statistics: the P1D of the overfluxes next to a measurement, and the flux
contrast term by term and order by order (Part D).

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()          # paper style (white); use_style("dark") for the dark one: same colour-blind-safe palette in both
from skewerjax.utils.plotting import (ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM, CMAPS, LABELS,
                                      FIGSIZES, add_panel_letters, plot_dag)
from skewerjax.core import LineBundle

In [ ]:
plot_dag(('lines', 'cosmology', 'skewers', 'samplers', 'fields', 'profiles', 'deposition', 'tau', 'overflux', 'stats',)); plt.show()

## Concepts

**Transmission and overflux.** $T = e^{-\tau}$ is the **transmission**; the flux is the observed $F = F_0\,T$, the
continuum $F_0$ times the transmission. The overflux follows DESI's continuum-fitting convention with the continuum known,

$$
\delta = \frac{F}{F_0\,\bar T} - 1 = \frac{T}{\bar T} - 1,\qquad \bar T(v) = \langle T\rangle_{\rm ens},
$$

with the ensemble mean taken at each velocity pixel $v$ over all skewers of all realisations. Each line $j$ has
$\delta_j = T_j/\bar T_j - 1$ with $\bar T_j(v) = \langle T_j\rangle_{\rm ens}$, and the total, $T = \prod_j T_j$, has
$\bar T = \langle\prod_j T_j\rangle_{\rm ens} \neq \prod_j \bar T_j$ (`summary_stats.DeltaFieldBuilder`; review notes,
Sec. Overflux, with the mean of each sample replaced by the ensemble mean at each pixel). Three consequences:

- $\langle\delta\rangle_{\rm ens} = 0$ holds exactly at every pixel (B1 prints it).
- The mean along a single skewer is not zero in general: a skewer keeps its own large-scale modes.
- In observed spectra, continuum fitting also removes the mean and the slope of each forest; a known continuum does not.

D2 splits $\delta_{\rm tot}$ exactly into the terms of the single lines and of their products, order by order.

**1D statistics.** Along skewers of $N$ pixels of width $dv$, P1D and $\xi_{\rm 1D}$ are an exact discrete Fourier pair
(review notes; `summary_stats.OverfluxStatistics1D`),

$$
P_{ab}(k) = \frac{dv}{N}\,\tilde\delta_a(k)\,\tilde\delta_b^*(k),\qquad
\xi_{ab}(\Delta v) = \frac{1}{N}\sum_j \delta_a(v_j)\,\delta_b(v_j + \Delta v),
$$

with $\tilde\delta$ the discrete Fourier transform, $k$ in s/km and $\Delta v$ in km/s, both averaged over the skewers.

**Figures.** A measurement is drawn as points with error bars joined by a thin line of the same colour, every
realisation (box) as a thin, faint line behind it, and an expectation as a line; the figures have no titles. In every
map the direction across the line of sight is horizontal and the line of sight vertical, from the origin at the
bottom-left corner; the measured iso-contours are green, with a transparent green band where the contour level lies
within the mean $\pm$ `N_SIGMA` times its error (the error bars of the 1D figures); the boundaries of the wedges of $\mu$
are white dash-dot rays, and a small coloured sector at the origin gives each wedge the colour it has in the wedge
figures.

**3D statistics on a grid (Parts A and C).** For one realisation $\delta$ on $N$ cells of volume $dV$ in a box of volume $V$,

$$
\hat P(\mathbf k) = \frac{\lvert\delta_{\mathbf k}\rvert^2}{V}\quad(\mathbf k \neq 0),\qquad
\hat\xi(\mathbf r) = \frac{1}{N}\sum_{\mathbf x} d(\mathbf x)\, d(\mathbf x + \mathbf r),\qquad d = \delta - \bar\delta_{\rm box},
$$

with $\delta_{\mathbf k}$ the discrete Fourier transform times $dV$ and $\mathbf r$ every minimum-image separation of the
periodic grid (the same normalisation as P1D, in 3D; `summary_stats.GridStatistics3D`). Both leave out the $\mathbf k = 0$
mode, the mean of the box, so they are one Fourier pair on the same modes. Both are averaged in bins: of
$\lvert\mathbf k\rvert$ or $\lvert\mathbf r\rvert$, of $(k_\perp, k_\parallel)$ or $(r_\perp, r_\parallel)$, and of
$\lvert\mathbf k\rvert$ or $\lvert\mathbf r\rvert$ in wedges of $\mu$ (A3), with the line of sight along $y$. Their
expectations, $P$ at every mode and $\xi_{\rm grid}$ at every separation (the toy model, below), are averaged over the
same modes and separations (A3). Every error bar is the error of the mean over the realisations of its bin alone. Part
A applies these estimators to the fields, Part C to the overfluxes on the lattice of skewers, both in comoving units.

**Overview.**

- **Part A — toy fields:** `n_real` periodic boxes with the spectrum and gas of the toy model below; $P$ and $\xi$ of the
  realisations against their expectation and against the continuous $P(k)$ and $\xi(r)$.
- **Part B — skewers:** a regular lattice of skewers along $y$ through every box (`SkewerRun`); $\tau$ of
  $\mathrm{Ly}\alpha$ and Si III, periodic, both on the velocity axis of the $\mathrm{Ly}\alpha$ transition, and the
  ensemble mean transmission at each pixel.
- **Part C — 3D statistics of the overfluxes:** the estimator checked on the density first; then $\xi$ and $kP$ of the
  total overflux in the $(\perp, \parallel)$ planes, $P$ and $\xi$ of the $\mathrm{Ly}\alpha$, Si III and total
  overflux, and $\xi$ of the total in wedges of $\mu$ next to a DESI measurement.
- **Part D — 1D statistics:** P1D of the overfluxes next to a DESI measurement; P1D and $\xi_{\rm 1D}$ of the total
  overflux on the $\mathrm{Ly}\alpha$ axis by order and term, averaged over the boxes.

**Configuration.** The boxes are configured in the cell after the toy model (`ToyBoxes`), and the cell after it prints
every value; the skewers (`SkewerRun`) are configured at the start of Part B. Everything is recomputed when the notebook
runs: nothing is read from or written to disk.

## The toy model: what `ToyBoxes` generates

**Grid and realisations.** $n_{\rm real}$ periodic cubes of side $L$ and volume $V = L^3$ on $N^3$ cells of side
$dx = L/N$; the modes are $\mathbf k = (2\pi/L)\,\mathbf n$ with integers $-N/2 < n_i \le N/2$. Realisation $i$ is drawn
from the seed `base_seed` $+\,i$ (`PowerLawGRF.sample`) and regenerated whenever it is needed: white noise $w$ on the
grid, $\delta_{\mathbf k} = dV\,\mathrm{FFT}[w]_{\mathbf k}\sqrt{P(k)/dV}$, so that
$\langle\lvert\delta_{\mathbf k}\rvert^2\rangle = V P(k)$, with $\delta_{\mathbf k = 0} = 0$.

**Power spectrum** (isotropic: no Kaiser or Fingers-of-God term), in the input parameters,

$$
P(k) = \underbrace{A_s\left(\frac{k}{k_0}\right)^{n_s - 1}}_{\text{power law}}
+ \overbrace{R_{\rm BAO}\,A_s}^{A_{\rm BAO}}\;
\underbrace{4\pi\,\overbrace{\sqrt{2\pi}\,\sigma_{\rm BAO}\,e^{-k^2\sigma_{\rm BAO}^2/2}}^{G(k)}
\left[\sigma_{\rm BAO}^2 + \frac{r_{\rm BAO}}{k}\right]}_{E(k)}\;
\left[1 + \underbrace{\frac{\sigma_{\rm BAO}^2\cos(k r_{\rm BAO}) + r_{\rm BAO}\sin(k r_{\rm BAO})/k}
{\sigma_{\rm BAO}^2 + r_{\rm BAO}/k}}_{f(k)}\right].
$$

Multiplied out, the BAO term is $A_{\rm BAO}E(1 + f) = 4\pi A_{\rm BAO}G\,[\sigma_{\rm BAO}^2(1 + \cos kr_{\rm BAO}) +
(r_{\rm BAO}/k)(1 + \sin kr_{\rm BAO})] \ge 0$ (`PowerLawGRF.bao_power`). $A_{\rm BAO}Ef$ is the exact Fourier transform
of a Gaussian bump in the correlation function,
$A_{\rm BAO}\bigl[e^{-(r - r_{\rm BAO})^2/2\sigma_{\rm BAO}^2} + e^{-(r + r_{\rm BAO})^2/2\sigma_{\rm BAO}^2}\bigr]$, and
its envelope $A_{\rm BAO}E$ a smooth, positive term peaked at $r = 0$.

**Amplitude.** $A_s$ is set by $\sigma_\delta$, the standard deviation of $\delta$ over the grid, without knowing $P$
first. At fixed $R_{\rm BAO}$ every term of $P$ is proportional to $A_s$: $P(k) = A_s\,p(k)$, where the shape $p(k)$
is $P$ at $A_s = 1$ and holds only $k_0$, $n_s$, $R_{\rm BAO}$, $\sigma_{\rm BAO}$ and $r_{\rm BAO}$. The variance of
the grid is the sum of $P$ over its modes, so it is $A_s$ times the same sum of the shape,

$$
\sigma_\delta^2 = \frac{1}{V}\sum_{\mathbf k \neq 0} P(k) = A_s\,S,\qquad
S = \frac{1}{L^3}\sum_{\substack{\mathbf n \neq 0\\ -N/2 < n_i \le N/2}} p\!\left(\frac{2\pi\lvert\mathbf n\rvert}{L}\right),
\qquad A_s = \frac{\sigma_\delta^2}{S}.
$$

$S$ is computed from the shape alone (`PowerLawGRF.variance` at $A_s = 1$), and $A_s$ is the ratio.

**Correlation function.** The expectation used throughout is the correlation of the discrete periodic field: the sum
over the modes of the box with $A_s$ substituted, next to its continuous counterpart over every $k$,

$$
\xi_{\rm grid}(\mathbf r) = \frac{1}{V}\sum_{\mathbf k \neq 0} P(k)\cos(\mathbf k\cdot\mathbf r)
= \sigma_\delta^2\,\frac{\sum_{\mathbf n \neq 0} p(2\pi\lvert\mathbf n\rvert/L)\,\cos(2\pi\,\mathbf n\cdot\mathbf r/L)}
{\sum_{\mathbf n \neq 0} p(2\pi\lvert\mathbf n\rvert/L)},
\qquad
\xi(r) = \frac{\sigma_\delta^2}{2\pi^2 S}\int_0^\infty k^2\,p(k)\,\frac{\sin kr}{kr}\,dk,
$$

with the same $\mathbf n$ in both sums, so $\xi_{\rm grid}(0) = \sigma_\delta^2$ exactly. $\xi_{\rm grid}$ lacks the
$k = 0$ mode and the modes beyond the Nyquist cube, and it is periodic; A3 shows what this changes.

**What sets $P$ and $\xi$.** Eight inputs (table). $L$ and $N$ enter because they fix the set of modes: through the sum
$S$ they set $A_s$, and with it the amplitude of $P(k)$; and $\xi_{\rm grid}$ is a sum over those very modes. The
continuous $P(k)$ and $\xi(r)$ depend on $L$ and $N$ only through $A_s$.

| Symbol | `ToyBoxes` argument | Meaning | Enters |
|---|---|---|---|
| $\sigma_\delta$ | `sigma_delta` | standard deviation of $\delta$ on the grid | $A_s = \sigma_\delta^2/S$ [cMpc$^3$] |
| $k_0$ | `k_0` | pivot wavenumber [cMpc$^{-1}$] | the shape $p$ |
| $n_s$ | `n_s` | spectral index: $P \propto k^{n_s - 1}$ | the shape $p$ |
| $R_{\rm BAO}$ | `bao_ratio` | $A_{\rm BAO}/A_s$, the height of the bump in $\xi$ per unit $A_s$ [cMpc$^{-3}$] | the shape $p$ |
| $\sigma_{\rm BAO}$ | `sigma_bao` | width of the BAO bump [cMpc] | the shape $p$ |
| $r_{\rm BAO}$ | `r_bao` | position of the BAO bump [cMpc] | the shape $p$ |
| $L$ | `box_cMpc` | box side [cMpc] | the modes: $S$ (so $A_s$) and $\xi_{\rm grid}$ |
| $N$ | `n_cells` | cells per side | the modes: $S$ (so $A_s$) and $\xi_{\rm grid}$ |

**Gas.** H I and Si III are one gas, cell by cell: $n_{\rm HI} = \bar n_{\rm HI}(1 + \delta)$,
$n_{\rm SiIII} = r\,n_{\rm HI}$, $T_{\rm gas} = T_0(1 + \delta)^{\gamma - 1}$ (written $T_{\rm gas}$ here, as $T$ is
the transmission), and peculiar velocities $\mathbf v = 0$ or linear theory, $\mathbf v_{\mathbf k} = i\,aHf\,
(\mathbf k/k^2)\,\delta_{\mathbf k}$. The remaining `ToyBoxes` inputs set this gas and its skewers, not $P$ or $\xi$:
`n_bar_HI` ($\bar n_{\rm HI}$, which sets $\langle T_{{\rm Ly}\alpha}\rangle$, B1), `siiii_to_hi` ($r$), `T0`,
`gamma_minus_1`, `velocity` (`"zero"`, the default, or `"linear"`), `f_growth` ($f$, for `"linear"` only), and `z`,
`Omega_m` ($a$ and $H$ of a flat cosmology: the Hubble flow, the velocity axis of the skewers); `n_real` and `base_seed`
choose the realisations.

**Positivity, $1 + \delta > 0$.** A Gaussian $\delta$ is unbounded, but a density $n = \bar n(1 + \delta)$ must be
positive (and $T_{\rm gas}$ needs $1 + \delta > 0$). The densities are not clipped; `ToyBoxes.fields` raises if any cell
has $\delta \le -1$. One cell falls below $-1$ with probability $\Phi(-1/\sigma_\delta)$ ($\Phi$ the standard normal
distribution function), and a box has $N^3$ cells, so about $N^3\,\Phi(-1/\sigma_\delta)$ of them do (printed below): a
finer grid needs a smaller $\sigma_\delta$. This has nothing to do with the spectrum: $P(k) \ge 0$ at every mode for any
$\sigma_\delta$, so the Gaussian field is always valid; only the mapping from $\delta$ to a density needs $1 + \delta > 0$.

In [ ]:
import time
from fractions import Fraction
from skewerjax.core import phys
from skewerjax.simulations import PowerLawGRF, ToyBoxes, field_statistics, xi_chi2, xi_subset_offsets
from skewerjax.simulations.toy_boxes import LINE_IDS
REALISATIONS = None               # None: all of them; e.g. [0, 1] for a quick check
L_BOX = 150
SIIII_TO_HI = 1 / 20 # n_SiIII = r n_HI
boxes = ToyBoxes(box_cMpc=L_BOX, n_cells=128, n_real=6, base_seed=42, bao_ratio=0.003, sigma_bao=1.0, r_bao=40, siiii_to_hi=SIIII_TO_HI, velocity="zero")

In [ ]:
# The values of the configuration (the text refers to them by their symbols)
from scipy.stats import norm
g = boxes.grf()
idx = np.arange(boxes.n_real) if REALISATIONS is None else np.asarray(REALISATIONS)
var_bao = 1.0 - float(PowerLawGRF(g.A_s, g.k_0, g.n_s, L=g.L, N=g.shape).variance() / g.variance())
S_shape = boxes.sigma_delta**2 / boxes.A_s                                       # the mode sum of the shape p(k), A_s = 1
print(f"dx = {boxes.dx:.4g} cMpc ({boxes.dx * boxes.aH:.1f} km/s); 1/L^3 = {S_shape:.4g} cMpc^-3")

## Part A — Toy fields and their statistics

### A1 · The fields

Every realisation of the toy model is generated, measured (A3) and discarded in one pass (`field_statistics`); the cell
also checks that $1 + \delta > 0$ in every cell of every realisation (the positivity paragraph of the toy model).

In [ ]:
res = field_statistics(boxes, realisations=REALISATIONS)        # P and xi of every realisation (A3), computed now
m1 = res["min_one_plus_delta"]
print(f"{res['n_real']} realisations measured in {res['time_s']:.0f} s, and the covariance of xi in {res['time_cov_s']:.0f} s: "
      f"min(1 + delta) = {m1.min():.3f} (realisation {res['realisations'][m1.argmin()]}); 1 + delta > 0 in {np.sum(m1 > 0)} of {m1.size}")

### A2 · The fields of one realisation

Realisation 0 in the plane of the first cell layer along the third axis (the cells with index $i_z = 0$), which holds
the first row of skewers of Part B: $x$ across and $y$, the line of sight of those skewers, upwards. The two columns
differ only by the factor $r$ of the densities; the temperature (not drawn) is a function of the density of the top row.
The bottom row shows the velocity along the skewers, $v_y$ (colour), and $(v_x, v_y)$ as arrows every `ARROW_EVERY`
cells; with zero velocities (`velocity="zero"`) it is empty and says so.

In [ ]:
from skewerjax.utils.plotting import plot_toy_fields
ARROW_EVERY = max(1, boxes.n_cells // 32)                                    # about 32 arrows across the slice
fields0 = boxes.fields(0)
fig, axs, info = plot_toy_fields(fields0, L_BOX, z_index=0, arrow_every=ARROW_EVERY)
add_panel_letters(axs.T.ravel(), labels=["a1", "a2", "b1", "b2"])
plt.show()

### A3 · The statistics of the realisations

Each realisation is measured on its own grid (`GridStatistics3D`). $P(\lvert\mathbf k\rvert)$ has one bin per mode for
the first modes, then bins of a quarter of the fundamental mode up to the end of the BAO wiggles,
$k \simeq 3/\sigma_{\rm BAO}$, then logarithmic bins; $\xi$ runs out to $r_{\rm BAO} + 4\sigma_{\rm BAO}$ when that lies
beyond $L/2$, so the whole peak shows (`ToyBoxes.bin_edges`). Panel (a) shows $k\,P(k)$, which flattens the steep
spectrum so the wiggles show; the BAO peak is shown as $r^2\xi(r)$ on linear axes.

**The expectation (black line), step by step.**

1. The generator draws every mode $\mathbf k$ of the box with $\langle\hat P(\mathbf k)\rangle = P(k)$, and the
   correlation at every separation $\mathbf r$ of the grid is, on average, $\xi_{\rm grid}(\mathbf r)$ (the toy model).
2. A bin $a$ of $\lvert\mathbf k\rvert$ holds the $N_a$ modes of the box that fall in it, so the expectation of the bin
   average is the input averaged over exactly those modes,
   $\bar P_a = N_a^{-1}\sum_{\mathbf k \in a} P(\lvert\mathbf k\rvert)$.
3. Likewise, a bin $a$ of $\lvert\mathbf r\rvert$ holds the $N_a$ separations of the grid that fall in it, and
   $\bar\xi_a = N_a^{-1}\sum_{\mathbf r \in a}\xi_{\rm grid}(\mathbf r)$, with
   $\xi_{\rm grid}(\mathbf r) = V^{-1}\sum_{\mathbf k \neq 0} P(k)\cos(\mathbf k\cdot\mathbf r)$
   (`GridStatistics3D.theory` on `boxes.grf().power_grid()`, the very array that scales the white noise).

**Why not the continuous $P(k)$ and $\xi(r)$ (thin grey lines).** They are the infinite-volume curves of the same
spectrum (`PowerLawGRF.power`, and `PowerLawGRF.correlation`, a numerical Hankel transform of the full $P$, BAO terms
included), and a finite periodic grid does not measure them:

- **No $k = 0$ mode.** $\xi_{\rm grid}$ integrates to zero over the box, so it lies below $\xi(r)$ by about the mean of
  $\xi$ over the box, a nearly constant offset (printed) that $r^2$ turns into a parabola; with a red spectrum it is large.
- **The Nyquist cube.** The grid has no modes beyond $\lvert k_i\rvert = \pi/dx$, so $\xi_{\rm grid}$ is smoothed on the
  scale of a cell, and the $\lvert\mathbf k\rvert$ shells near $\pi/dx$ are cut by the cube's faces.
- **Periodic images.** $\xi_{\rm grid}(\mathbf r)$ adds the correlation of the images $\mathbf r + L\mathbf n$, which
  matters near $L/2$ (the wedges below).
- **Averaging within bins.** $P$ and $\xi$ vary inside a bin (the wiggles, the peak), and the few modes of a low-$k$
  bin sample it unevenly, so a bin average is not the curve at the bin centre.

The binned expectation includes all of this, so it is exact bin by bin; the continuous curves are the reference only.

**The error bars** are the error of the mean over the $n_{\rm real}$ realisations, bin by bin:
$\sigma_a/\sqrt{n_{\rm real}}$, with $\sigma_a$ the standard deviation (with $n_{\rm real} - 1$) of the $n_{\rm real}$
values of bin $a$. The residual panels divide by it.

In [ ]:
from skewerjax.utils.plotting import plot_isotropic_recovery, plot_kpar_kperp_maps, plot_rpar_rperp_maps
fig, axs, rec = plot_isotropic_recovery(res, xi_view="r2", r_bao=boxes.r_bao, grf=g)
add_panel_letters(axs[0])                                              # (a) k P, (b) r^2 xi; the residual panels below them
plt.show()
SUBSETS = (4, 8, 16)                                                   # disjoint subsets of realisations (reading below)
XI_WINDOW = (2 * boxes.r_bao / 3, boxes.r_bao + 4 * boxes.sigma_bao)   # the r range of the offset [cMpc]
c_all, c_win = xi_chi2(res), xi_chi2(res, r_range=XI_WINDOW)

*Reading the figure.* The measurement follows the BAO wiggles of $k\,P$ bin by bin (a), and the peak of $r^2\xi$ at
$r_{\rm BAO}$ (b). In (b) the points can sit on one side of the expectation over a whole range of $r$, by more than their
error bars. This is one ordinary fluctuation, not a bias:

- **The $\xi$ bins move together.** Every bin of $\xi$ is a sum over the same modes,
  $\hat\xi(\mathbf r) = V^{-1}\sum_{\mathbf k}\hat P(\mathbf k)\cos(\mathbf k\cdot\mathbf r)$, and with a red spectrum
  the few largest-scale modes carry most of the signal. When their random amplitudes come out a little high (or low),
  every bin moves up (or down) together. The error bars show the scatter of each bin alone, so they cannot show this:
  several points off by about $1\sigma$ in the same direction are one fluctuation, not several.
- **The numbers say so.** The exact covariance $C$ of the bins (`GridStatistics3D.xi_iso_covariance`) is dominated by
  one pattern, its first eigenvector (its share of the variance is printed). The $\chi^2$ that accounts for it,
  $\chi^2 = \mathbf d^{\rm T}(C/n_{\rm real})^{-1}\mathbf d$ with $\mathbf d = \bar\xi - \xi_{\rm grid}$, is close to
  the number of bins, over all the bins and over the window of the BAO peak, `XI_WINDOW` (printed, with the probability
  of a larger value); the offset lies almost entirely along the three largest patterns. The covariance itself is
  right: the $\chi^2$ of each realisation averages the number of bins.
- **More realisations shrink it, without a sign.** For disjoint subsets of `SUBSETS` realisations (when there are that
  many), the offset of $r^2\xi$ in `XI_WINDOW` changes sign from subset to subset and stays within its standard
  deviation, which falls as $1/\sqrt n$ (printed).
- **$P$ shows no such effect**, because its bins hold different modes, which are independent.

The grey continuous $\xi(r)$ lies far above both: the missing $k = 0$ mode lowers $\xi_{\rm grid}$ by a nearly constant
amount (printed), which $r^2$ magnifies; the grey $k\,P(k)$ runs through the binned expectation, except in the first
bins, which hold a few modes each.

**The $(k_\perp, k_\parallel)$ and $(r_\perp, r_\parallel)$ planes.** The same statistics resolved across ($x$ axis) and
along ($y$ axis) the line of sight: in each figure, (a) the expectation and (b) the mean measured. $k\,P$ on a
logarithmic colour scale with both axes logarithmic; $r^2\xi$ on a linear colour scale, linear axes up to half the box.
Three iso-contour levels, the same for the expectation (dashed, in both panels) and the measurement (green, with its
band of $\pm$ `N_SIGMA` errors). The white rays are the boundaries of the wedges of $\mu$ of the next figure
(`MU_WEDGES`), and the coloured sectors at the origin their colours there.

In [ ]:
fig, axs, chi2_P = plot_kpar_kperp_maps(res, n_sigma=1)
add_panel_letters(axs)
plt.show()
print(f"chi2/dof per bin: k P map {chi2_P:.2f}")

In [ ]:
fig, axs, chi2_xi = plot_rpar_rperp_maps(res, r_bao=boxes.r_bao, n_sigma=0.5)
add_panel_letters(axs)
plt.show()
print(f"chi2/dof per bin: xi map {chi2_xi:.2f}")
# The ring and what lies beneath it, wedge by wedge at r_BAO: the bump = xi - xi_nw (xi_nw: the expectation without the
# BAO wiggle) and the background xi_nw
a = res["axes"]
j = np.argmin(np.abs(a["r_mu"] - boxes.r_bao), axis=1)
w = np.arange(len(j))
th, nw = res["theory"]["xi_mu"][w, j], res["theory_nw"]["xi_mu"][w, j]
meas, err = res["xi_mu"][w, j] - nw, res["xi_mu_err"][w, j]
print(f"at r = r_BAO, in units of A_BAO = {g.A_bao:.3g}:")
print("  wedge of mu      bump expected   bump measured     background xi_nw")
for k in w:
    print(f"  {a['mu_edges'][k]:.2f} - {a['mu_edges'][k + 1]:.2f}       {(th[k] - nw[k]) / g.A_bao:6.3f}      "
          f"{meas[k] / g.A_bao:6.2f} +- {err[k] / g.A_bao:4.2f}      {nw[k] / g.A_bao:+6.2f}")
k_nyq = np.pi / boxes.dx
print(f"P_BAO at the Nyquist frequency pi/dx = {k_nyq:.2f} cMpc^-1, relative to its value at the fundamental mode: "
      f"{float(g.bao_power(k_nyq) / g.bao_power(2 * np.pi / L_BOX)):.1e}")

**Wedges.** The direction of a separation (or of a wavevector) is measured by

$$
\mu = \frac{r_\parallel}{\lvert\mathbf r\rvert} = \cos\theta,
$$

the cosine of its angle $\theta$ to the line of sight: $\mu = 1$ along the line of sight, $\mu = 0$ across it.
$\mathrm{Ly}\alpha$ 3D analyses show the anisotropy in wedges of $\mu$: $r^2\xi(r)$ against $r$ on linear axes, the
measurement as points with error bars in one colour per wedge.
The expectation of a wedge $w$ (an interval of $\mu$ of width $\Delta\mu$), step by step:

1. In the continuum, the correlation depends on the separation and its direction, $\xi(r, \mu)$, and the wedge average
   is $\xi_w(r) = \Delta\mu^{-1}\int_{\mu \in w}\xi(r, \mu)\,d\mu$. The toy field is isotropic, $\xi(r, \mu) = \xi(r)$,
   so every wedge would be the same curve.
2. On the grid, the code averages $\xi_{\rm grid}(\mathbf r)$ over the separations of the grid with
   $\lvert\mathbf r\rvert$ in the bin $a$ and $\mu = \lvert r_y\rvert/\lvert\mathbf r\rvert$ in the wedge $w$:
   $\bar\xi_{a w} = N_{aw}^{-1}\sum_{\mathbf r \in a,\,\mu \in w}\xi_{\rm grid}(\mathbf r)$.
3. $\xi_{\rm grid}$ is not isotropic, so the dashed lines differ between wedges. In a periodic box, with the modes cut
   off by a cube rather than a sphere,

   $$
   \xi_{\rm grid}(\mathbf r) \simeq \sum_{\mathbf n \in \mathbb Z^3}\xi\big(\lvert\mathbf r + L\mathbf n\rvert\big) - \bar\xi_{\rm box},
   $$

   the sum over the periodic images of $\mathbf r$, minus the mean over the box (the missing $k = 0$ mode). The nearest
   image is closer along the axes than along the diagonals: at $\lvert\mathbf r\rvert = r$ along an axis it lies at
   $L - r$, which is $L/2$ at $r = L/2$, while along a body diagonal it lies at about $0.82L$. So the wedge near
   $\mu = 1$, along the $y$ axis, differs most from the others near $L/2$. Beyond $L/2$ a wedge only holds the
   separations that still fit in the cube.

In [ ]:
from skewerjax.utils.plotting import plot_xi_mu_wedges
fig, axs, chi2_rmu = plot_xi_mu_wedges(res, r_bao=boxes.r_bao)
plt.show()
print("chi2/dof per wedge (diagonal):", np.round(chi2_rmu, 2))

## Part B — Skewers

### B1 · Skewers through each box

**Layout.** In every box, skewers run along $y$ through the centres of the cell columns (the nodes $x = i_x\,dx$,
$z = i_z\,dx$ of `grf_cells`), from $y = 0$ to $y = L$, so their first and last points are the same place of the periodic
box. `SkewerBundle`, the cells of `ToyBoxes.cells` and `compute_optical_depth` give $\tau$ of $\mathrm{Ly}\alpha$ and
Si III 1206, periodic over one box (`velocity_period=True`) and both on the $\mathrm{Ly}\alpha$ velocity units reference frame axis
(`line_frame="reference"`).

**The set-up, `SkewerRun`** (next cell, which prints what follows from it), with $p$ = `points_per_cell` and
$N_{\rm pix}$ = `n_pix`:

| Argument | What it controls | Result or cost |
|---|---|---|
| `stride` $= (s_x, s_z)$ | the cell columns that become skewers: every $s_x$-th in $x$ and every $s_z$-th in $z$, from column 0, exactly through the cell centres $x = i\,s_x\,dx$, $z = j\,s_z\,dx$; $\lceil N/s_x\rceil \lceil N/s_z\rceil$ skewers per box, $s\,dx$ apart | the result: the number of skewers (the noise of every statistic) and the transverse sampling of Part C (Nyquist frequency $\pi/(s\,dx)$); the cost is linear in the number of skewers |
| `points_per_cell` | $p$, the points of the path per cell length along $y$, $N p + 1$ per skewer: every cell is cut into pieces of width $W = dx/p$ | the result until it converges ($\tau$ sums the pieces at their centroids, accurate while $W \ll b$: the resolution check below); the cost of $\tau$ is linear in $p$ |
| `n_pix` | $N_{\rm pix}$ velocity pixels over one period, of width $dv = v_{\rm per}/N_{\rm pix}$: the velocity resolution | the result: the $k$ range of P1D, up to $\pi/dv$, and the line-of-sight resolution of Part C; the cost of $\tau$ is linear in $N_{\rm pix}$ |
| `batch` | the skewers per call of `cell_pieces` and `compute_optical_depth` | the memory only, never the result |

Here $v_{\rm per}$ is the velocity span of one box along a skewer, and $u = v_{\rm per}/L$ converts comoving distances to
velocities (Part C).

In [ ]:
from skewerjax.simulations import SkewerRun, run_skewers
SETUP = SkewerRun(stride=(4, 4), points_per_cell=16, n_pix=128)     # the skewers (table above)
der = SETUP.derived(boxes)
n_x, n_z = der["n_x"], der["n_z"]
cols = boxes.columns(SETUP.stride)
print(f"skewers: stride {tuple(SETUP.stride)} -> {n_x} x {n_z} = {der['per_box']} per box, {der['per_box'] * idx.size} in all "
      f"({idx.size} boxes)")
print(f"transverse spacing of the skewers: stride x dx = {SETUP.stride[0]} x {boxes.dx:.4g} = {der['d_perp_cMpc'][0]:.4g} cMpc in x")
print(f"points per skewer: {der['points_per_skewer']} ({SETUP.points_per_cell} per cell); pieces of W = {boxes.dx / SETUP.points_per_cell:.4f} cMpc "
      f"= {der['piece_kms']:.3f} km/s")
print(f"pixels: {SETUP.n_pix} over v_per = {der['period_kms']:.0f} km/s: dv = {der['dv_kms']:.2f} km/s, "
      f"P1D up to pi / dv = {np.pi / der['dv_kms']:.4f} s/km")
print(f"u = v_per / L = {der['kms_per_cMpc']:.2f} km/s per cMpc (aH = {der['aH_start']:.2f} at y = 0, {der['aH_end']:.2f} at y = L)")

**Two passes, in memory** (`run_skewers`). First, each box is generated and $\tau$ computed along its skewers; its
$\tau$ is kept, and the density on the lattice of skewers is measured with the 3D estimator of Part C (C0). Then the
ensemble mean transmission at each pixel, $\bar T_j(v)$ of each line and $\bar T(v)$ of the total, is taken over every
skewer of every box, and the overfluxes of every box are formed against it and measured (the 3D statistics of Part C
and the 1D statistics of Part D). Only $\tau$ and the statistics are held, never the fields; nothing is stored on disk.

In [ ]:
t0 = time.perf_counter()
run = run_skewers(boxes, SETUP, realisations=REALISATIONS)              # every box, in memory
t_skewers = time.perf_counter() - t0
n_box, n_sk = run["flux"]["n_real"], len(cols)
dev = {d: run["time_s"][run["device"] == d].sum(1) for d in np.unique(run["device"])}
period = float(run["skewers"][0]["period"])
lines = LineBundle(list(LINE_IDS), ref_line_id=LINE_IDS[0])
b_T0 = {lid: np.sqrt(2 * phys.kB * boxes.T0 / m) / 1e3 for lid, m in zip(LINE_IDS, np.asarray(lines.mass_amu))}
print(f"{n_box} boxes x {n_sk} skewers ({n_x} x {n_z}) in {t_skewers:.0f} s")
print("time per box: " + ", ".join(f"{d} {t.mean():.0f} s ({t.size} boxes)" for d, t in dev.items())
      + f"; per box: fields {run['time_s'][:, 0].mean():.1f} s, pieces {run['time_s'][:, 1].mean():.1f} s, "
      f"tau {run['time_s'][:, 2].mean():.0f} s, statistics {run['time_s'][:, 3].mean():.1f} s")
print(f"axis: period {period:.0f} km/s, {SETUP.n_pix} pixels of dv = {period / SETUP.n_pix:.2f} km/s; a cell spans "
      f"{period / boxes.n_cells:.1f} km/s ({SETUP.n_pix / boxes.n_cells:g} pixels), a piece {period / boxes.n_cells / SETUP.points_per_cell:.2f} "
      f"km/s; b at T0: " + ", ".join(f"{lid} {b:.1f} km/s" for lid, b in b_T0.items()))
res_check = run["resolution"]
print(f"resolution check: max W / b = {np.round(res_check[..., 0].max(axis=(0, 1)), 3)}, column fraction above b/4 at most "
      f"{np.round(res_check[..., 1].max(axis=(0, 1)), 6)} (Lya, Si III); warnings: {sum(len(w) for w in run['warnings'].values())}")
print(f"mean transmission <T>: Lya {run['mean_F'][:, 0].mean():.4f}, Si III {run['mean_F'][:, 1].mean():.4f}, total "
      f"{run['mean_F_total'].mean():.4f} (scatter between boxes {run['mean_F'][:, 0].std():.1e}); T_bar(v) of Lya from "
      f"{run['mean_transmission'][0, 0]:.4f} at the start of the skewers to {run['mean_transmission'][0, -1]:.4f} at the end")
print(f"ensemble normalisation: max over the pixels of |<delta>_ens| = {np.abs(run['mean_delta']).max():.1e} (Lya, Si III, total); "
      f"max |[sum_A delta_A - C] / (1 + C) - delta_tot| = {run['decomposition_error']:.1e}")

### B2 · One skewer, from the gas to the flux contrast

**Where the skewers run.** The H I density of realisation 0 in the plane of A2 (the first cell layer along the third
axis, $i_z = 0$), a plane that contains the line of sight and the first row of skewers: the skewers of that row are the
thin black lines, with small arrows along their direction ($+y$), $s_x\,dx$ apart (printed in B1, marked at the top).
The thick line is the skewer of the next figure.

In [ ]:
from skewerjax.utils.plotting import plot_skewer_slice, plot_toy_skewer
sk = run["skewers"][0]
on_z0 = np.flatnonzero(sk["columns"][:, 1] == 0)
j = int(on_z0[np.argmin(np.abs(sk["columns"][on_z0, 0] - boxes.n_cells // 2))])   # the column nearest to x = L/2 with i_z = 0
ix, iz = sk["columns"][j]
fig, ax, sl_info = plot_skewer_slice(fields0, L_BOX, sk["columns"], z_index=0, highlight=(ix, iz))
plt.show()
print(f"{sl_info['n_row']} skewers in the layer i_z = 0, {sl_info['spacing_cMpc']:.4g} cMpc apart; the skewer of the next figure: "
      f"x = {ix * boxes.dx:.1f} cMpc")

**Along the skewer.** Rows (a) and (b) are along the comoving position $y$ (the $N$ cell values, as steps); rows (c) and
(d) along the velocity of the $\mathrm{Ly}\alpha$ axis, on which Si III is evaluated at $v + \Delta v$ and wrapped into
the period. The overdensities (b) of H I and Si III are identical by construction (one gas, with one temperature);
their absorption is not: Si III is narrower ($b \propto m^{-1/2}$) and displaced by the separation $\Delta v$ of the two
lines, wrapped into the period (printed below). Row (c) is the transmission $T = e^{-\tau}$ of each line and the total
$T = \prod_j T_j$; row (d) their overfluxes, $\delta = T/\bar T - 1$, each against its ensemble mean $\bar T(v)$ at every
pixel (B1), so the mean of (d) along this one skewer need not vanish. The very thin grey vertical lines are the grids:
in (a) and (b) the points of the path ($p$ per cell), in (c) and (d) the pixel centres, where $\tau$ is evaluated. When
the lines of a panel would be too dense to read, they are drawn only in a zoomed inset, whose range is marked on the
panel.

In [ ]:
n_col = {sp: fields0["n"][sp][ix, :, iz] for sp in ("HI", "SiIII")}
fig, axs, grid_info = plot_toy_skewer(L_BOX, n_col, sk["v"], sk["tau"][:, j], mean_transmission=run["mean_transmission"],
                                      points_per_cell=SETUP.points_per_cell)
add_panel_letters(axs)
plt.show()
dv_12 = float(lines.delta_v[1])
dv_fold = abs((dv_12 + period / 2) % period - period / 2)                    # the separation folded into half the period
print(f"Si III - Lya separation Delta v = {dv_12:.0f} km/s, wrapped into the period of {period:.0f} km/s: "
      f"{(dv_12 + period / 2) % period - period / 2:.0f} km/s; b(Lya) / b(Si III) = {b_T0[LINE_IDS[0]] / b_T0[LINE_IDS[1]]:.2f}")

## Part C — 3D statistics of the overfluxes

The overflux of each line and of the total, $\delta_X = T_X/\bar T_X(v) - 1$ against its ensemble mean at each pixel
(Concepts), on the lattice of skewers of Part B is a 3D field, periodic in the three directions, measured with the
estimators of Part A.

**Two kinds of axis.** Along the line of sight the coordinate is a **velocity**, $v$ [km/s]; across it, the coordinates
are comoving **distances**, $x_\perp$ [cMpc]. One approximate factor relates them, the mean Hubble rate along a skewer
($dv/dy = aH$ changes along it with $a(y)$; B1 prints $u$ and $aH$ at both ends):

$$
\Delta v \simeq u\,\Delta x,\qquad u = \frac{1}{L}\int_0^L aH\,dy = \frac{v_{\rm per}}{L}.
$$

Two rules follow (`utils.plotting.comoving_axis`, `velocity_axis`):

- **Maps** ($r_\parallel$ against $r_\perp$, $k_\parallel$ against $k_\perp$): each axis keeps its native coordinate.
  The bottom axis is the perpendicular one, in cMpc (cMpc$^{-1}$ for $k$); the left axis is the line of sight, in km/s
  (s/km for $k$); the origin is the bottom-left corner. For reference only, a top axis gives the perpendicular
  coordinate in km/s and a right axis the line of sight in cMpc, both through $u$.
- **Isotropic quantities that combine both axes** ($\lvert\mathbf r\rvert$, $\lvert\mathbf k\rvert$, $\mu$, $r^2\xi$,
  $kP$): the line of sight is converted to comoving distance, $r_\parallel = v/u$ and $k_\parallel = u\,k_v$, and they are
  reported in comoving units: $r$ in cMpc, $r^2\xi$ in cMpc$^2$, $k$ in cMpc$^{-1}$ (the top axes give km/s and s/km for
  reference). On the lattice a pixel $dv$ spans $dv/u = L/N_{\rm pix}$ cMpc, so the lattice is a cube of side $L$.

**What the lattice resolves.** Along the line of sight: from the pixel size $dv$ up to the span of the box, $v_{\rm per}$
($k_\parallel$ from $2\pi/v_{\rm per}$ to $\pi/dv$, i.e. from $2\pi/L$ to $\pi N_{\rm pix}/L$ in cMpc$^{-1}$). Across it:
from the skewer spacing $s\,dx$ (set by `stride`) up to $L$ ($k_\perp$ from $2\pi/L$ to $\pi/(s\,dx)$).

**Errors.** Points with error bars are the mean over the boxes and its error, as in Part A, with every box as a faint
line. The overflux has no exact expectation to draw.

### C0 · The estimator, checked on the density

Before the overflux, the density overdensity of the same boxes, sampled on the lattice of skewers (every cell along
$y$), goes through exactly the same estimator (`SkewerRun.grid3d`, the same bins in cMpc). Its exact expectation is
$\xi_{\rm grid}$ at the separations of the lattice, whose spectrum is the input folded over the aliases of the
transverse lattice, $P_{\rm lat}(\mathbf k) = \sum_{\mathbf m} P(\mathbf k + 2\pi\mathbf m/(s\,dx))$
(`lattice_density_theory`). With few boxes the scatter between boxes is itself a noisy error, so this check uses the
exact Gaussian errors: $\bar P_a\sqrt{2/(N_a\,n_{\rm real})}$ for a bin of $N_a$ modes, and the exact covariance of the
$\xi$ bins (A3). Printed: $\chi^2$ of $P$ and of $\xi$, $\chi^2$ of $\xi$ against the expectation without the BAO wiggle,
and the height of the BAO peak of $r^2\xi$ above the straight line through its sidebands at
$r_{\rm BAO} \pm 4\sigma_{\rm BAO}$ (`bao_height`). The tests also check the estimator against a direct pair sum over the
coordinates of the lattice, with minimum-image separations (`tests/summary_stats/test_statistics.py`).

In [ ]:
from skewerjax.simulations import bao_height, lattice_density_check
chk = lattice_density_check(run, boxes, SETUP)
d3 = run["density3d"]
print(f"the density on the lattice ({chk['n_real']} boxes) through the estimator of the overflux: P(|k|) chi2 = {chk['P_chi2']:.1f} "
      f"for {chk['P_dof']} bins; xi(|r|) chi2 = {chk['xi_chi2']:.1f} for {chk['xi_dof']} bins (full covariance), and "
      f"{chk['xi_chi2_nw']:.0f} against the expectation without the BAO wiggle")
b, be, bn = chk["bao"], chk["bao_expected"], chk["bao_nw"]
print(f"BAO peak of r^2 xi at r = {b['r']:.1f} cMpc above its sidebands: measured {b['height']:.3f} +- {b['height_err']:.3f}, "
      f"expected {be['height']:.3f}, without the wiggle {bn['height']:.3f} cMpc^2")

### C1 · The $(r_\perp, r_\parallel)$ and $(k_\perp, k_\parallel)$ planes of the total overflux

$r^2\xi(r_\perp, r_\parallel)$ [cMpc$^2$] of the total overflux, as the measured panel of the maps of A3: a linear,
diverging colour scale centred on 0, three iso-$r^2\xi$ contours with their band of $\pm$ `N_SIGMA` errors, and the
wedges of $\mu$ of C3. $r_\parallel$ is binned in steps of three cells along the skewers and $r_\perp$ in the transverse
spacing of the lattice, both up to half the box. The diamonds on the line of sight ($r_\perp = 0$) mark where the
Si III–$\mathrm{Ly}\alpha$ cross term puts its features: $r_\parallel = \Delta v$ (open), and its BAO echo at
$\Delta v + u\,r_{\rm BAO}$ and $\lvert u\,r_{\rm BAO} - \Delta v\rvert$ (thin), each drawn only when it lies inside the map.

In [ ]:
from skewerjax.utils.plotting import plot_overflux_3d, plot_overflux_kpar_kperp, plot_overflux_rpar_rperp
u = period / L_BOX                                                       # km/s per cMpc (B1): v = u x, k_v = k_x / u
ax3, f3 = run["axes3d"], run["flux3d"]                                   # comoving bins [cMpc], [cMpc^-1]
lim = SETUP.limits(boxes)
NAMES = {"HI-LyA": r"Ly$\alpha$", "SiIII-1206": "Si III 1206", "total": "total"}
tot = dict(f3["total"], axes=ax3)                                        # the total overflux and its bins
fig, ax, marks = plot_overflux_rpar_rperp(tot, kms_per_cMpc=u, r_bao=boxes.r_bao, delta_v=dv_fold, n_sigma=1)
plt.show()
dr_par, r_par_max = np.diff(ax3["rpar_edges"])[0], ax3["rpar_edges"][-1]
print(f"bins: r_par {dr_par:.2f} cMpc = {u * dr_par:.0f} km/s, r_perp {np.diff(ax3['rperp_edges'])[0]:.2f} cMpc (the transverse "
      f"spacing), out to {r_par_max:.1f} cMpc = {u * r_par_max:.0f} km/s; u r_BAO = {u * boxes.r_bao:.0f} km/s")
for name, y in (("Delta v", dv_fold), ("Delta v + u r_BAO", dv_fold + u * boxes.r_bao), ("|u r_BAO - Delta v|", abs(u * boxes.r_bao - dv_fold))):
    print(f"  marker {name} = {y:.0f} km/s: {'drawn' if y <= u * r_par_max else 'beyond the map, not drawn'}")
X, Xe = f3["total"]["xi_map"], f3["total"]["xi_map_err"]
ip = int(np.argmin(np.abs(ax3["rpar"] - dv_fold / u)))                   # the r_par bin of Delta v
for q in (0, 1):                                    # r_perp bins: 0 = pairs on the same skewer, 1 = one transverse spacing
    excess = X[ip, q] - 0.5 * (X[ip - 2, q] + X[ip + 2, q])
    print(f"Si III - Lya at r_par = Delta v, r_perp bin {q} ({ax3['rperp_edges'][q]:.1f}-{ax3['rperp_edges'][q + 1]:.1f} cMpc): "
          f"xi = {X[ip, q]:.2e}, excess over the bins two away {excess:.2e} = {excess / Xe[ip, q]:.1f} sigma")
tau_ly = sk["tau"][0].mean(0)                                            # the mean tau of Lya along the skewers of box 0
Tt0 = np.exp(-sk["tau"]).prod(0)
shared_box = (Tt0 / Tt0.mean() - 1).mean(0)                    # the pattern every skewer of box 0 shares, normalised by its box mean
shared_ens = (Tt0 / run["mean_transmission"][-1] - 1).mean(0)  # the same against the ensemble mean at each pixel
print(f"along a skewer: Delta ln a = v_per / c = {period / phys.c_kms:.3f}; mean tau_Lya {tau_ly[:5].mean():.3f} at the start, "
      f"{tau_ly[-5:].mean():.3f} at the end (box 0)")
print(f"the pattern shared by the skewers of box 0: variance {shared_box.var():.2e} against the mean of the box (the sawtooth), "
      f"{shared_ens.var():.2e} against the ensemble mean at each pixel (the box's own k_perp = 0 modes); over every box: "
      f"{np.abs(run['mean_delta'][-1]).max():.0e}")
R2 = ax3["rpar"][:, None] ** 2 + ax3["rperp"][None, :] ** 2
far = R2 > (L_BOX / 4) ** 2
x_d, x_t = (R2 * d3["xi_map"])[far], (R2 * X)[far]
print(f"beyond r = L / 4, the map against r^2 xi of the density of the same boxes on the same lattice (C0): correlation "
      f"coefficient {np.corrcoef(x_d, x_t)[0, 1]:.2f}, ratio {np.sum(x_d * x_t) / np.sum(x_d**2):.2f} (P_tot / P_density in the "
      f"first |k| bin {f3['total']['P_iso'][0] / d3['P_iso'][0]:.2f})")
for name, (i, k) in (("r_perp = 0, largest r_par", (-1, 0)), ("r_par = 0, largest r_perp", (0, -1))):
    print(f"  {name}: total {R2[i, k] * X[i, k]:+.2f}, density {R2[i, k] * d3['xi_map'][i, k]:+.2f}, its expectation "
          f"{R2[i, k] * d3['theory']['xi_map'][i, k]:+.2f} cMpc^2")

### C2 · Power spectrum and correlation function of the overfluxes

The counterpart of the $k\,P$ and $r^2\xi$ figure of A3, with its estimators and styles, in comoving units, over the
ranges where they are well defined:

- **$\lvert\mathbf k\rvert \le k_{\rm N} = \min\bigl(\pi N_{\rm pix}/L,\ \pi/(s_x\,dx),\ \pi/(s_z\,dx)\bigr)$**, the
  smallest Nyquist frequency of the lattice ($\pi N_{\rm pix}/L = u\,\pi/dv$ along the line of sight). A
  $\lvert\mathbf k\rvert$ bin averages a shell of modes, and only inside the sphere of radius $k_{\rm N}$ does every
  shell hold modes in every direction. Beyond it, the coarser axes have no modes left, so the shell keeps only the
  directions near the finer axis, and $P(\lvert\mathbf k\rvert)$ would turn into the power along that one direction.
- **$\lvert\mathbf r\rvert \le L/2$.** The lattice is periodic, so is its correlation function,
  $\xi(\mathbf r + L\hat{\mathbf e}_i) = \xi(\mathbf r)$: a separation beyond half the box along an axis is the same as
  its image inside, and adds nothing new. Below the transverse spacing the $\lvert\mathbf r\rvert$ bins hold separations
  along the line of sight only.

In [ ]:
fig, axs = plot_overflux_3d(f3, ax3, kms_per_cMpc=u, r_bao=boxes.r_bao, names=NAMES)
add_panel_letters([axs["P"], axs["xi"]])
plt.show()
PT = f3["total"]["P_iso"]
print(f"lattice {n_x} x {SETUP.n_pix} x {n_z} per box, a cube of side L = {L_BOX:g} cMpc (v_per = {period:.0f} km/s along the line of sight)")
print(f"|k| from {ax3['k_edges'][0]:.3f} to k_N = {ax3['k_edges'][-1]:.3f} cMpc^-1 in {ax3['k_edges'].size - 1} bins: Nyquist "
      f"pi N_pix / L = {lim['k_nyq_par']:.3f} cMpc^-1 along the line of sight (pi / dv = {lim['k_nyq_par'] / u:.2e} s/km), "
      f"pi / (s dx) = {lim['k_nyq_perp'][0]:.3f} (x), {lim['k_nyq_perp'][1]:.3f} (z) cMpc^-1 across it")
print(f"|r| up to L / 2 = {lim['r_max']:.0f} cMpc in {ax3['r_edges'].size - 1} bins of {np.diff(ax3['r_edges'])[0]:.2f} cMpc; "
      f"line-of-sight separations only below the transverse spacing {lim['r_perp_min']:.2f} cMpc")
print("P_line / P_tot at the lowest |k|: " + ", ".join(f"{f} {f3[f]['P_iso'][0] / PT[0]:.2f}" for f in LINE_IDS))
r3 = ax3["r_iso"]
print(f"BAO peak of r^2 xi above its sidebands at r_BAO -+ 4 sigma_BAO [cMpc^2], against (P_X / P_density at the lowest |k|) "
      f"x the density's expected {chk['bao_expected']['height']:.3f}:")
for f in (*LINE_IDS, "total"):
    bf = bao_height(r3, r3**2 * f3[f]["xi_iso"], boxes.r_bao, boxes.sigma_bao, r3**2 * f3[f]["xi_iso_err"])
    print(f"  {NAMES[f]:>12}: r^2 xi(r_BAO) = {bf['value']:.3f}, height {bf['height']:.3f} +- {bf['height_err']:.3f}; "
          f"scaled density {f3[f]['P_iso'][0] / d3['P_iso'][0] * chk['bao_expected']['height']:.3f}")

*Reading the figure.*

- **$k\,P$.** The BAO wiggles of the density show in the overflux of every line. At low $k$ the total holds more power
  than its two lines together (their ratios add up to less than one, printed): H I and Si III are one gas, so their
  overfluxes are correlated, and their cross term, $\propto\cos(k_\parallel\Delta v)$, adds in phase where
  $k_\parallel \ll 2\pi/\Delta v$. At larger $\lvert\mathbf k\rvert$ every shell averages the cross term over all
  directions, which washes out its oscillation in $k_\parallel$ (the bands of the map above; it shows in P1D, D1).
  Si III's share is set by its abundance $r$ and its mean transmission (B1).
- **$r^2\xi$.** The BAO peak shows at $r_{\rm BAO}$ in every overflux. On these scales the overflux follows the density,
  $\xi_X \simeq b_X^2\,\xi_\delta$, and the height of its peak above the sidebands agrees with $b_X^2$ (the ratio of the
  spectra at the lowest $\lvert\mathbf k\rvert$) times the density's within the errors (printed). The errors are those of
  `n_real` boxes: large at large $r$, where few modes contribute, and correlated between bins (A3), so the smooth rise
  to $r_{\rm BAO}$ and the fall beyond it are common to all the bins. Against the mean of a whole box, the shared sawtooth
  (C1) added to every $\xi$ a term of order its variance, which $r^2$ turned into a broad hump that hid the peak.

### C3 · Wedges of the total overflux

The wedges of A3 for the total overflux, in comoving units ($r$ [cMpc], $r^2\xi$ [cMpc$^2$], km/s on the top axis),
measured only: the overflux has no expectation. The open squares are measured $\mathrm{Ly}\alpha \times \mathrm{Ly}\alpha$
wedges (`utils.reference_data.XI_LYA_LYA_AA_WEDGES`, passed as `reference=`; converted from $h^{-1}$Mpc to cMpc with
$h = H_0/100$ of the toy cosmology: $r/h$ and $r^2\xi/h^2$), in the colour of the wedge with the same $\mu$ edges and
labelled with their own $\mu$ ranges. They are a realistic reference only: our toy total overflux (with Si III) is not
calibrated to them.

In [ ]:
from skewerjax.utils.reference_data import XI_LYA_LYA_AA_WEDGES as XI_REF
fig, axs, _ = plot_xi_mu_wedges(tot, r_bao=boxes.r_bao, kms_per_cMpc=u, reference=XI_REF)
plt.show()
h = phys.H0_km_s_mpc / 100
mu_e, r_mu = ax3["mu_edges"], ax3["r_mu"]
jb = np.nanargmin(np.abs(r_mu - boxes.r_bao), axis=1)
print(f"reference: r [h^-1 Mpc] / h and r^2 xi [h^-2 Mpc^2] / h^2 with h = {h:g}; up to r = {XI_REF['r'].max() / h:.0f} cMpc")
print("r^2 xi of the total at r_BAO [cMpc^2]: " + "; ".join(
      f"{mu_e[w]:.2f} < mu < {mu_e[w + 1]:.2f}: {r_mu[w, j]**2 * f3['total']['xi_mu'][w, j]:+.2f} +- {r_mu[w, j]**2 * f3['total']['xi_mu_err'][w, j]:.2f}"
      for w, j in enumerate(jb)))

*Reading the figure.* The four wedges of the total overflux agree within their errors at every $r$: with zero
velocities the toy field is isotropic, and the sawtooth of a box-mean normalisation, which split the wedges, is gone
(C1). The wedge along the line of sight ($\mu > 0.95$) also carries the Si III–$\mathrm{Ly}\alpha$ spot at
$\Delta v/u$, and it rises highest at large $r$, where it follows the density of these boxes near the line of sight and
the periodic image along the $y$ axis (A3, C1); its errors are the largest, as it holds the fewest separations. The
measured $\mathrm{Ly}\alpha$ wedges behave differently: along the line of sight they turn negative, which the toy model,
with no peculiar velocities and a known continuum, does not reproduce.

## Part D — 1D statistics

P1D and $\xi_{\rm 1D}$ along the skewers. D1 sets the P1D of the overfluxes of Part C next to a DESI measurement; D2 and
D3 decompose the total overflux on the $\mathrm{Ly}\alpha$ axis into its orders and terms, with lines 1 =
$\mathrm{Ly}\alpha$ (H I) and 2 = Si III 1206. Every overflux is taken against its ensemble mean transmission at each
pixel (Concepts). Every curve is the mean over the boxes and its error the error of the mean
over the boxes (the skewers of one box share its modes): points with error bars joined by thin lines, with every box as
a faint line behind them. P1D is binned with one mode per bin for the first modes and logarithmic bins after,
$\xi_{\rm 1D}$ in bins of a few pixels, up to half the period, in km/s with cMpc on the top axis.

### D1 · P1D next to a DESI measurement

$k_\parallel P_{\rm 1D}(k_\parallel)/\pi$ against $k_\parallel$ [s/km], linear in $k_\parallel$ and logarithmic in $P$:
the convention and the $k_\parallel$ range of Fig. 1 of Chaves-Montero et al. (DESI DR1 $\mathrm{Ly}\alpha$ P1D,
`.project_material/Chaves-Montero`, `data_journal.tex`). The toy model's P1D is the mean over the skewers of
$(dv/N_{\rm pix})\lvert\tilde\delta(k_\parallel)\rvert^2$ of the overfluxes of Part C (against the ensemble mean at
each pixel), binned to the spacing of the
measurement (mean and error over the boxes), for the total overflux and for each line; the grey squares are the
measurement at the lowest redshift of that figure (`utils.reference_data.P1D_DESI_DR1_Z22`). The top axis gives
$k_\parallel$ in cMpc$^{-1}$ with the factor $u$ of Part C. This is a fiducial visual reference only: the toy model is
not calibrated to data, and it is at a single redshift $z$, not that of the measurement.

In [ ]:
from skewerjax.utils.plotting import plot_p1d_reference
from skewerjax.utils.reference_data import P1D_DESI_DR1_Z22 as P1D_REF
fig, ax = plot_p1d_reference(run["p1d"], P1D_REF, names=NAMES, k_max=P1D_REF["k_max_figure"], kms_per_cMpc=u)
plt.show()
kc = run["p1d"]["k"]
toy = np.interp(P1D_REF["k"], kc, kc * run["p1d"]["p1d"][0] / np.pi)
print(f"toy z = {boxes.z:g}, reference z = {P1D_REF['z']:g}; reference / toy (Lya) over the reference's k range: "
      f"median {np.median(P1D_REF['kP_pi'] / toy):.0f}")
print(f"Si III - Lya separation Delta v = {abs(dv_12):.0f} km/s: period 2 pi / Delta v = {2 * np.pi / abs(dv_12):.4f} s/km; "
      f"Nyquist frequency of the cells along the skewers pi / (u dx) = {np.pi / (u * boxes.dx):.4f} s/km")

*Reading the figure.* The toy model has the shape of the measurement, $k_\parallel P_{\rm 1D}/\pi$ rising at low
$k_\parallel$ and then flattening, but not its amplitude: its fluctuations are those of a Gaussian field of small
$\sigma_\delta$, without non-linear structure, so its P1D lies far below the data (printed). The total oscillates in
$k_\parallel$ with period $2\pi/\Delta v$ (printed): the cross term of a shifted copy,
$2\cos(k_\parallel\Delta v)\sqrt{P_{11}P_{22}}$, swings it between $(\sqrt{P_{11}} \pm \sqrt{P_{22}})^2$, a visible
modulation even for a much weaker Si III, as $\sqrt{P_{22}}$ enters. The $k_\parallel$ axis ends at the Nyquist frequency
of the pixels, $\pi/dv$ (B1), before that of the cells along the skewers, $\pi/(u\,dx)$ (printed), and far before the
thermal cut-off of either line, so neither cut-off shows. The measurement shows oscillations of the same origin, metal
lines overlapping the $\mathrm{Ly}\alpha$ forest (the paper's Sec. 2.1), much weaker: the IGM's Si III is far less
abundant than the toy's.

### D2 · The decomposition by order

**The terms.** `DeltaFieldBuilder` also forms, per skewer, the products $\delta_A = \prod_{j \in A}\delta_j$ of every
subset $A$ of lines. With $\mathcal C(v)$, the ensemble means of the products evaluated at each pixel,
$\mathcal C = \langle\delta_1\delta_2\rangle_{\rm ens}$ for two lines (plus the higher orders for more), the total overflux
decomposes **exactly** into them (review notes, Sec. Overflux, with the ensemble mean at each pixel):

$$
\delta_{\rm tot} = \frac{\sum_j\delta_j + \sum_{j_1 < j_2}\delta_{j_1}\delta_{j_2} + \cdots - \mathcal C}{1 + \mathcal C},\qquad
\mathcal C(v) = \sum_{\lvert A\rvert \ge 2}\langle\delta_A\rangle_{\rm ens}(v) = \frac{\bar T(v)}{\prod_j \bar T_j(v)} - 1.
$$

It follows from $T = \prod_j \bar T_j(1 + \delta_j)$ and its ensemble mean,
$\bar T = \prod_j \bar T_j\,\bigl(1 + \sum_j\langle\delta_j\rangle_{\rm ens} + \mathcal C\bigr)$. The order-1 term of the
denominator, $\sum_j\langle\delta_j\rangle_{\rm ens}$, vanishes only because the normalisation is ensemble-wide: each
$\bar T_j$ is the mean over the same ensemble at the same pixel (with a mean per skewer or per box it would not). The sum
of the terms reproduces $\delta_{\rm tot}$ at every pixel of every skewer to machine precision (printed).
`OverfluxStatistics1D` gives P1D and $\xi_{\rm 1D}$ of the total, of every pair of terms $(A, B)$ and of every order
$\lvert A\rvert + \lvert B\rvert$: order $n$ collects the terms with $\lvert A\rvert + \lvert B\rvert = n$, order 0 is the
normalisation term $\mathcal C$, and the orders add up to the total to machine precision (printed).

**The terms of two lines.** With two lines, at each pixel,

$$
T = \bar T_1 \bar T_2\, (1 + \delta_1)(1 + \delta_2),\qquad
\delta = \frac{\delta_1 + \delta_2 + \delta_1\delta_2 - \mathcal C}{1 + \mathcal C},\qquad \mathcal C = \langle \delta_1 \delta_2 \rangle_{\rm ens}.
$$

Every term of $\xi_{\rm 1D}$ (real space) or P1D (Fourier space) is a pair of subsets $A, B$ of $\{1, 2\}$ (or the
normalisation term), of order $\lvert A\rvert + \lvert B\rvert$:

| $A \backslash B$ | $\mathcal{C}$ (order 0) | 1 (order 1) | 2 (order 1) | 12 (order 2) |
|---|---|---|---|---|
| $\mathcal{C}$ | $+\mathcal{C}\mathcal{C}$ | 0 | 0 | $-\mathcal{C}\mathcal{C}$ |
| 1 | 0 | $\xi_{11}$, $P_{11}$ | $\xi_{21}$, $P_{21}$ | $\xi_{(12)1}$, $P_{(12)1}$ |
| 2 | 0 | $\xi_{12}$, $P_{12}$ | $\xi_{22}$, $P_{22}$ | $\xi_{(12)2}$, $P_{(12)2}$ |
| 12 | $-\mathcal{C}\mathcal{C}$ | $\xi_{1(12)}$, $P_{1(12)}$ | $\xi_{2(12)}$, $P_{2(12)}$ | $\xi_{(12)(12)}$, $P_{(12)(12)}$ |

The zeros are exact after the ensemble average, because $\langle\delta_j\rangle_{\rm ens} = 0$ at every pixel. For a
$\mathcal C$ constant along the skewer, the ensemble average gives

$$
\xi = \frac{-\mathcal{C}^2 + \xi_{11} + \xi_{22} + \xi_{12,{\rm sym}} + \xi_{1(12),{\rm sym}} + \xi_{2(12),{\rm sym}} + \xi_{(12)(12)}}{(1 + \mathcal C)^2},
$$

$$
P(k) = \frac{P_{11} + P_{22} + P_{12,{\rm sym}} + P_{1(12),{\rm sym}} + P_{2(12),{\rm sym}} + P_{(12)(12)} - 2\pi\,\mathcal C^2\, \delta_D(k)}{(1 + \mathcal C)^2},
$$

where "sym" adds the pair and its transpose: 1/2 with 2/1, 1/12 with 12/1, and 2/12 with 12/2. $\mathcal C(v)$ varies
slowly along the skewer, with the mean transmission, so the code weights every term at its own pixel by
$1/(1 + \mathcal C(v))$, which keeps the decomposition exact, and the constant pieces only touch the lowest $k$.

For a second line that is a shifted copy of the first, $\delta_2(v) \propto \delta_1(v - \Delta v)$, the cross term is
$P_{12,{\rm sym}} \propto 2\cos(k\Delta v)\, P_{11}$. It oscillates with period $2\pi/\Delta v$ in $k$ (D1, and
the cross terms of D3): the Fourier image of the $\xi_{\rm 1D}$ peak at $\Delta v$.

**By order.** Order 2 holds the Gaussian-like products and dominates; orders 3 and 4 are the non-linear corrections of
the product of the two fields. The line separation $\Delta v$, folded into the period (dashed; B2 prints both), is where
the cross terms put a peak in $\xi_{\rm 1D}$; in P1D they oscillate as $\cos(k\Delta v)$, with period $2\pi/\Delta v$
in $k$ (printed below in units of the mode spacing). The top axes give $k_\parallel$ and the separation in comoving units
(factor $u$).

In [ ]:
from skewerjax.utils.plotting import plot_orders_compared, plot_terms_compared
ens = {"total overflux": run["flux"]}
dv_lines = [0.0, dv_12]
fig, axs = plot_orders_compared(ens, "P", delta_v_list=dv_lines, kms_per_cMpc=u)
add_panel_letters(axs.T.ravel())
plt.show()
fig, axs = plot_orders_compared(ens, "xi", delta_v_list=dv_lines, kms_per_cMpc=u)
add_panel_letters(axs.T.ravel())
plt.show()
e = run["flux"]
print(f"max |sum of the orders - total| / max |total|: P {np.abs(e['P_order'].sum(0) - e['P_total']).max() / np.abs(e['P_total']).max():.1e}, "
      f"xi {np.abs(e['xi_order'].sum(0) - e['xi_total']).max() / np.abs(e['xi_total']).max():.1e}; every pixel: "
      f"max |[sum_A delta_A - C] / (1 + C) - delta_tot| = {run['decomposition_error']:.1e}; order 1 / max |total|: "
      f"P {np.abs(e['P_order'][1]).max() / np.abs(e['P_total']).max():.1e}")
print(f"cos(k Delta v): period 2 pi / Delta v = {2 * np.pi / abs(dv_12):.2e} s/km = "
      f"{period / abs(dv_12):.2f} mode spacings (2 pi / period)")

### D3 · Each order, term by term

The same decomposition one level down, one row per order. Every term $(A, B)$ of an order is drawn with its transpose
$(B, A)$ as A–B (their sum, which is real), on the order's total (the thick line). Order 2 holds the products 1–1, 2–2
and 1–2 (the cross term) and the normalisation cross term $\mathcal{C}$–12; orders 3 and 4 are 1–12, 2–12 and 12–12. The
order-0 term $\mathcal{C}$–$\mathcal{C}$ is not shown: it is nearly constant in $\xi_{\rm 1D}$, and lives at the lowest
$k$ in P1D.

In [ ]:
fig, axs = plot_terms_compared(ens, "P", kms_per_cMpc=u)
add_panel_letters(axs.ravel())
plt.show()
fig, axs = plot_terms_compared(ens, "xi", kms_per_cMpc=u)
for ax in axs.ravel():
    ax.axvline(dv_fold, color=DIM, ls=":", lw=1.2)                       # the separation, folded
add_panel_letters(axs.ravel())
plt.show()

*Reading the figures.* Every term follows the density field, scaled (at linear order the
overflux is $-\bar\tau$ times the density contrast, a bias); the thermal widths of both lines (B1) are narrower than a pixel, so their
cut-off lies beyond the $k$ range. The cross terms 1–2 carry the line separation, a property of the axis, not of the
gas: a peak at $\Delta v$ in $\xi_{\rm 1D}$, an oscillation of period $2\pi/\Delta v$ in P1D. With zero velocities
(`velocity="zero"`) the flux is in real space, apart from the thermal widths; `velocity="linear"` moves the absorption by
the peculiar velocities (redshift space).

The kernels of `skewerjax.summary_stats` (`p1d_kernel`, `GridStatistics3D`, `binned_mean`) are jitted; the checks
are in `tests/summary_stats/test_statistics.py` and `tests/simulations/test_toy_boxes.py`.